# 00 – Setup check

**Goal:** Verify that the environment is ready: Python packages, PostgreSQL connection and database schema.
**Input:** none · **Output:** none (diagnostics only)
**Assessment items:** P3 (database with SQL queries from Python), B2 (PostgreSQL), B6 (public GitHub repo)

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: /workspaces/f1-overtaking-analytics


## 1. Python packages

In [2]:
from importlib.metadata import version

packages = ["fastf1", "pandas", "numpy", "sqlalchemy", "psycopg", "statsmodels",
            "scikit-learn", "xgboost-cpu", "shap", "geopandas", "folium", "rapidfuzz"]
print(f"Python {sys.version.split()[0]}")
for pkg in packages:
    print(f"  {pkg:<14} {version(pkg)}")

Python 3.11.15
  fastf1         3.9.0
  pandas         2.3.3
  numpy          2.4.6


  sqlalchemy     2.1.6
  psycopg        3.3.6
  statsmodels    0.15.0
  scikit-learn   1.9.1
  xgboost-cpu    3.2.0
  shap           0.51.0
  geopandas      1.2.0
  folium         0.20.0
  rapidfuzz      3.14.6


## 2. Database connection

In [3]:
import pandas as pd
from sqlalchemy import text
from src.db import get_engine, init_schema, table_counts

engine = get_engine()
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## 3. Schema

`init_schema` is idempotent: it creates missing tables and leaves existing data untouched.

In [4]:
tables = init_schema(engine)
pd.Series(table_counts(engine), name="rows").to_frame()

,rows
circuits,23
driver_map,42
drivers,26
laps,30165
pairs,9425
races,28
results,560
weather,4491


In [5]:
# Column overview straight from the PostgreSQL catalog
pd.read_sql(
    """
    SELECT table_name, COUNT(*) AS n_columns,
           STRING_AGG(column_name, ', ' ORDER BY ordinal_position) AS columns
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name
    ORDER BY table_name
    """,
    engine,
)

,table_name,n_columns,columns
0,circuits,11,"circuit_id, name, locality, country, lat, lon,..."
1,driver_map,4,"season, abbreviation, driver_number, driver_id"
2,drivers,5,"driver_id, code, given_name, family_name, nati..."
3,laps,13,"race_id, driver_id, lap_number, lap_time_s, ti..."
4,pairs,18,"pair_id, race_id, lap_number, driver_ahead, dr..."
5,races,8,"race_id, season, round, race_name, circuit_id,..."
6,results,7,"race_id, driver_id, team, grid, quali_pos, fin..."
7,weather,8,"race_id, time_s, air_temp, track_temp, humidit..."
